In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import OneHotEncoder
import joblib

# Load cleaned data
data = pd.read_csv("../analysis/wfp_food_prices_ken_cleaned.csv")

# Convert date to datetime and extract features
data['date'] = pd.to_datetime(data['date'])
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month

market_input = pd.DataFrame({'market': ['Nairobi', 'Kisumu']})

# Encode market as categorical
encoder = OneHotEncoder(sparse_output=False)
market_encoded = encoder.fit_transform(data[['market']])
market_cols = encoder.get_feature_names_out(['market'])
market_df = pd.DataFrame(market_encoded, columns=market_cols)

#add volatility column
data['volatility'] = data['price'].rolling(3).std().fillna(0).round(2)

# Combine features
X = pd.concat([data[['year', 'month', 'volatility']], market_df], axis=1)
y = data['price']

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Initialize Random Forest
model = RandomForestRegressor(
    n_estimators=500, 
    max_depth=10,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print("Random Forest trained successfully")
print("Mean Absolute Error:", mae)

# Predict based on user input
year_input = int(input("Enter the year: "))
month_input = int(input("Enter the month (1-12): "))
market_input = input(f"Enter market ({', '.join(data['market'].unique())}): ")

# Encode market input
market_input_df = pd.DataFrame({'market':[market_input]})
market_array = encoder.transform(market_input_df)
market_df_future = pd.DataFrame(market_array, columns=market_cols)

#calculate average volatility
volatility_input = data['volatility'].mean()

# Combine features
X_future = pd.concat([pd.DataFrame([[year_input, month_input, volatility_input]], columns=['year','month']),
                      market_df_future], axis=1)
#display the predicted price
predicted_price = model.predict(X_future)
print(f"Predicted maize price in {market_input} for {month_input}/{year_input}: {predicted_price[0]:.2f}")

#save model and encoder
joblib.dump(model, "../model/rf_food_price_model.pkl")
joblib.dump(encoder, "../model/market_encoder.pkl");


Random Forest trained successfully
Mean Absolute Error: 14.16639790773256


Enter the year:  2089
Enter the month (1-12):  6
Enter market (Nairobi, Kisumu):  Kisumu


ValueError: 2 columns passed, passed data had 3 columns